# Analysis 

Here we will take you through an analysis of different models, after having trained models with nested fold approach.

[IN PROGRESS]

In [ ]:
## setup ##
from NM_TinyRNN.code.measures import analysis
from NM_TinyRNN.code.measures import mechanistic_variability as mech_var
from NM_TinyRNN.code.measures import reversal_metrics as revme
from NM_TinyRNN.code.measures import plotting_stats

from NM_TinyRNN.code.models import nested_jobs # this is the code used to train models!

import numpy as np
import pandas as pd
import torch #for testing a few things
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from importlib import reload

CODE_DIR = Path('.') ## OBS THIS MAY NEED TO BE ADJUSTED!
SAVE_PATH = CODE_DIR/'NM_TinyRNN/data/rnns'
DATA_PATH = Path('./NM_TinyRNN/data/AB_behaviour/')

%load_ext autoreload


We will perform analyses nd plotting primarily from pandas dataframes.
First we get an analysis_df, which contains a row for each model.
Each row has columns for
```python
'info_dict_path','trials_data_path','model_state_path'
``` 
 that we use to point to files and load data when running analyses.

Pandas dataframes also let us easily subset our data using .query(), so for this reason the analysis_df also has columns for 
```python
'model_type','hidden_size','nonlinearity','constraint'.
```

In [ ]:
#construct analysis_df and have a look at a subset of data!
info_df = nested_jobs.get_job_info_df()
all_models_df = analysis.get_analysis_df(info_df, mode='all')

subset_df = all_models_df.query('subject_ID == "WS16" and model_id == "2_unit_monoGRU_relu_unipolar"')

subset_df

### Performance

Here we want to summarise the held-out performance of different models, but also 

In [ ]:
## SOME SUBJECTS HAVE MORE PREDICTABLE BEHAVIOUR THAN OTHERS

subject_df = all_models_df.groupby(['subject_ID','model_id','outer_loop_n',
                                    'model_type','hidden_size',
                                    'nonlinearity']).mean('eval_CE')
fig, ax = plt.subplots(2,1, figsize = (10,6), sharey=True)
ax[0].set_title('variance: outer and inner')

all_models_df = all_models_df.sort_values(by=['subject_ID','model_type'])
sns.stripplot(all_models_df.query('hidden_size==2'), x='model_type',y='eval_CE', 
              hue = 'subject_ID', legend = True, dodge=True,
              ax=ax[0], )
ax[1].set_title('variance: outer (mean across inner)')

subject_df = subject_df.sort_values(by=['subject_ID','model_type'])
sns.stripplot(subject_df.query('hidden_size==2'), x='model_type',y='eval_CE', 
              hue = 'subject_ID', legend = False, dodge=True,
              ax=ax[1])
sns.move_legend(ax[0], "upper left", bbox_to_anchor=(1, 1))

fig.tight_layout()


In [ ]:
reload(plotting_stats)

tanh_models = all_models_df.query('nonlinearity=="tanh"')
plotting_stats.plot_paired_comparison(tanh_models,y='eval_CE',
                                      x='hidden_size',
                                      within_variable='model_type', 
                                      paired_across='subject_ID',
                                      mean_across='outer_loop_n',)

plt.show()


relu_models = all_models_df.query('nonlinearity=="relu"')
plotting_stats.plot_paired_comparison(relu_models,y='eval_CE',
                                      x='hidden_size',
                                      within_variable='model_type', 
                                      paired_across='subject_ID',
                                      mean_across='outer_loop_n',)

plt.show()



In [ ]:
# is tanh performing better than RELU?


plotting_stats.plot_paired_comparison(all_models_df.query('hidden_size==2'),y='eval_CE',
                                      x='nonlinearity',
                                      within_variable='model_type', 
                                      paired_across='subject_ID',
                                      mean_across='outer_loop_n',)

plt.show()

In [ ]:
#Is there a difference for 2-unit relu models across architectures?
reload(plotting_stats)
plotting_stats.plot_repeated_measures_anova(subject_df.query('hidden_size==2 and nonlinearity=="relu"'),
                                            y='eval_CE',
                                      x='model_type',
                                      paired_across='subject_ID',
                                      mean_across='outer_loop_n',)

plt.show()

### Reversals

testing/developing code to plot data for reversals

In [ ]:

trials_data = analysis.load_data(info_df.iloc[2500].trials_data_path)
trials_data = revme.add_reversal_columns(trials_data)

fig, ax = plt.subplots()
sns.violinplot(trials_data.query('reversal_type=="RL"'),
                x='reversal_trial_index',
                y='prob_B',
                ax =ax)

### Mechanisms


In [ ]:
## Have a look at the mechanisms across folds !

subset_df = all_models_df.query('model_id=="2_unit_monoGRU_relu_unipolar" and subject_ID=="WS16"')

subset_df = subset_df.sort_values(by=['outer_loop_n','eval_CE'])

fig, ax = plt.subplots(10,9,figsize=(30,30), sharex=True, sharey=True)
axes = ax.flatten()
for i,each_row in enumerate(subset_df.itertuples()):
    
    trials_data = analysis.load_data(each_row.trials_data_path)
    sns.scatterplot(trials_data, x= 'logit_value',y='logit_change',
                    hue = 'trial_type',
                    legend = False, ax = axes[i])
    axes[i].set_title(f'outer {each_row.outer_loop_n}. eval_CE:{each_row.eval_CE:.3f}')
fig.suptitle(f'Subject: {each_row.subject_ID}\n model: {each_row.model_id}')
fig.tight_layout()

In [ ]:
## what about mechanisms? are they more or less similar across models architectures?

reload(mech_var)
sim_df = compute_similarities(all_models_df.query('hidden_size == 2'))
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
sns.stripplot(data=all_models_df, y='model_id', x='eval_CE', ax=ax[0])
sns.stripplot(data=sim_df.query('component=="hidden"'), 
              y='model_id', x='similarity', ax=ax[1])
sns.stripplot(data=sim_df.query('component == "gate_update"'), 
              y='model_id', x='similarity', ax=ax[2])
ax[0].set(title='performance'); ax[1].set(title='hidden states'); ax[2].set(title='gating mechanism')
plt.tight_layout()
plt.show()